In [40]:
from ingest import load_faq_data
documents = load_faq_data()

In [73]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [74]:
documents = documents_llm

In [78]:
doc = documents[0]
print(doc["doc_id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [45]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]
    
    
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [133]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
ollama_client = OpenAI(
    base_url="https://api.openai.com/v1",
    api_key="api"
)

In [47]:
import json

user_prompt = json.dumps(doc)

In [48]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [134]:
response = ollama_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [135]:
result = response.output_parsed

print(result)

print(result.questions)

questions=['I just found this course, am I still allowed to join now?', 'Can I start the course late, or is it too late to enroll?', 'If I join after the course has already started, can I still get a certificate?', 'What do I need to do to be eligible for the certificate if I’m joining now?', 'Is it okay to discover the course late and still complete the project in time for a certificate?']
['I just found this course, am I still allowed to join now?', 'Can I start the course late, or is it too late to enroll?', 'If I join after the course has already started, can I still get a certificate?', 'What do I need to do to be eligible for the certificate if I’m joining now?', 'Is it okay to discover the course late and still complete the project in time for a certificate?']


In [60]:
from evaluation_utils import llm_structured

In [207]:
result, usage = llm_structured(
    ollama_client,
    data_gen_instructions,
    user_prompt,
    Questions,
)

print(result.questions)

['Can I still join the course if I just found it now?', 'If I join late, can I still get a certificate somehow?', 'What do I need to do to be eligible for a certificate in this course?', 'Is it okay to start the course after it has already begun?', 'Until when can I submit my project if I want the certificate?']


In [63]:
usage.input_tokens, usage.output_tokens

(1727, 9)

In [64]:
from evaluation_utils import calc_price

In [65]:
cost = calc_price(usage)

cost

{'input_cost': 0.00129525, 'output_cost': 4.05e-05, 'total_cost': 0.00133575}

In [66]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[]

In [67]:
from evaluation_utils import llm_structured_retry

In [137]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        ollama_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model="gpt-5.4-mini"
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["doc_id"]
        })

    return results, usage

In [138]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [86]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [139]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [140]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [142]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08818574999999998

In [143]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08818574999999998

In [144]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [146]:
df_ground_truth.to_csv("ground_truth-new.csv", index=False)

In [148]:
import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [149]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [150]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [151]:
q = ground_truth[0]
q

{'question': 'I just found this course — is it still okay to join now?',
 'document': '74eb249bbf'}

In [152]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [154]:
relevance = []

for d in results:
    relevance.append(int(d["doc_id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [158]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["doc_id"] == doc_id))

    return relevance

In [159]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

I just found this course — is it still okay to join now?


[1, 0, 0, 0, 0]

In [160]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

How can I watch the live workshop stream if I’m not an instructor or TA?


[1, 0, 0, 0, 0]

In [161]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where do I follow the LLM Zoomcamp syllabus and see the deadlines and homework in one place?


[1, 0, 0, 0, 0]

In [162]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [163]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [164]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [165]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["doc_id"] == doc_id))

    return relevance

In [166]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [167]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [168]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

In [170]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [171]:
cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

14

In [172]:
cnt / len(example)
# 0.933

0.9333333333333333

In [173]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [174]:
hit_rate(example)
# 0.933

0.9333333333333333

In [175]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

12.333333333333332

In [176]:
total_score / len(example)
# 0.822

0.8222222222222222

In [177]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [178]:
mrr(example)
# 0.822

0.8222222222222222

In [179]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [180]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.8654867256637168, 'mrr': 0.7296460176991149}

In [181]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [182]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/565 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8938053097345132, 'mrr': 0.8067551622418876}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.9026548672566371, 'mrr': 0.7954572271386426}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8654867256637168, 'mrr': 0.7296460176991149}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8336283185840708, 'mrr': 0.697728613569321}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7964601769911505, 'mrr': 0.6619174041297934}


In [186]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [187]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

In [188]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
6,1.0,4.0,0.1,0.971681,0.878525
7,1.0,4.0,0.2,0.971681,0.877286
3,1.0,2.0,0.1,0.966372,0.877021
19,2.0,4.0,0.2,0.966372,0.877021
35,5.0,10.0,0.5,0.966372,0.877021
4,1.0,2.0,0.2,0.962832,0.875959
18,2.0,4.0,0.1,0.964602,0.875044
20,2.0,4.0,0.5,0.962832,0.874985
34,5.0,10.0,0.2,0.966372,0.874602
33,5.0,10.0,0.1,0.966372,0.874071


In [189]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [192]:
import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [193]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [195]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["doc_id"]] = doc